## Data - 01 - Import

## Setup

In [1]:
import os
import json
import shutil
import pandas as pd

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
ORIG_DIR = os.path.join(ROOT, "orig")
DATA_DIR = os.path.join(ROOT, "data")
OUT_DIR = os.path.join(ROOT, "output")

for d in [ORIG_DIR, DATA_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

RAW_JSON = os.path.join(ROOT, "data", "output", "exams_raw.json")
RAW_COPY = os.path.join(ORIG_DIR, "exams_raw.json")

if os.path.isfile(RAW_JSON) and not os.path.isfile(RAW_COPY):
    shutil.copy2(RAW_JSON, RAW_COPY)

with open(RAW_COPY, "r", encoding="utf-8") as f:
    papers = json.load(f)

rows = []
for paper in papers:
    paper_name = paper.get("paper")
    page_count = paper.get("page_count")
    for p in paper.get("pages", []):
        rows.append({
            "paper": paper_name,
            "page": p.get("page"),
            "method": p.get("method"),
            "length": p.get("length") if "length" in p else p.get("char_count"),
            "page_count": page_count,
            "text": p.get("text", "")
        })

## Dataset

In [3]:
df = pd.DataFrame(rows)

print(df.shape)
df.head(19)

(187, 6)


,paper,page,method,length,page_count,text
0,Discrete_Mathematics_2013_Final_E.pdf,1,text,740,12,1 \n \n WIT \n \n \n \nBACHELOR OF SC...
1,Discrete_Mathematics_2013_Final_E.pdf,2,text,924,12,2 \n \n \n \nQuestion 1 \n \n(a) \nThe univers...
2,Discrete_Mathematics_2013_Final_E.pdf,3,text,632,12,3 \n \n \n \nQuestion 1 continued \n(c) \n \n(...
3,Discrete_Mathematics_2013_Final_E.pdf,4,text,34,12,4 \n \nQuestion 2 continued overleaf
4,Discrete_Mathematics_2013_Final_E.pdf,5,text,860,12,5 \n \nQuestion 2 continued \n \n(b) \n(1) Us...
5,Discrete_Mathematics_2013_Final_E.pdf,6,text,1072,12,6 \n \nQuestion 3 \n \n(a) \n(i) \nHow many d...
6,Discrete_Mathematics_2013_Final_E.pdf,7,text,345,12,7 \n \nQuestion 4 \n \n(a) \n \n(i) \nDraw the...
7,Discrete_Mathematics_2013_Final_E.pdf,8,text,556,12,8 \n \nQuestion 4 continued \n \n(b) \n(i) \n...
8,Discrete_Mathematics_2013_Final_E.pdf,9,text,368,12,9 \n \nQuestion 4 continued \n \n \n(iii) Usi...
9,Discrete_Mathematics_2013_Final_E.pdf,10,text,656,12,10 \n \nPLEASE ASK FOR THE NEW MATHEMATICS TAB...


## Cleaning
- Removing blank pages
- Repeating headers + footers
- Distinguish between exam papers and marking schemes
- Fix OCR words
- Identify question structure A), i), marks etc.
- Topics + keywords
- Removing "\n"

# Removing blank pages + pages with no questions

- Any pages that have less then 50 characters on that page are removed such as Question continued
- Any pages that have like formulas on it usualy the pages at the end

In [4]:
df.groupby("method")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
ocr,43.0,710.581395,402.336937,322.0,411.00,497.0,966.0,1875.0
text,144.0,914.083333,418.804267,1.0,702.75,901.5,1063.0,2615.0


In [5]:
df[df["length"] < 50]

,paper,page,method,length,page_count,text
3,Discrete_Mathematics_2013_Final_E.pdf,4,text,34,12,4 \n \nQuestion 2 continued overleaf
61,Discrete_Mathematics_2017_Final_E.pdf,7,text,1,15,6


In [6]:
df[(df["paper"] == "Discrete_Mathematics_2017_Final_E.pdf") & (df["page"] == 7)]

,paper,page,method,length,page_count,text
61,Discrete_Mathematics_2017_Final_E.pdf,7,text,1,15,6


In [7]:
df = df[df["length"] >= 50].reset_index(drop=True)

In [8]:
df[df["length"] < 50]

,paper,page,method,length,page_count,text


In [9]:
df = df[~df["text"].str.contains("Python Cheat Sheet", na=False)]
df = df[~df["text"].str.contains("Laws of Logic", na=False)]
df = df[~df["text"].str.contains("Table of Logical", na=False)]

In [10]:
print(df["text"].str.contains("Python Cheat Sheet", na=False).any())
print(df["text"].str.contains("Laws of Logic", na=False).any())
print(df["text"].str.contains("Table of Logical", na=False).any())

False
False
False


# Repeating headers and footers

- make a list of all the exam paper stuff that repeats like the first page how it has whatever like setu or wit discrete maths exam but keep the year becasue that will be a column im keeping the year and when the exam was becasue those are important for picking exams in that year and for predicting whether a question will come up.

In [11]:
import re
def extract_metadata(text):
    year_match = re.search(r"\b(DECEMBER|AUGUST)\s+\d{4}\b", text)
    semester_match = re.search(r"(SEMESTER\s+\d+\s*-\s*YEAR\s+\d+)", text)

    year = year_match.group(1) if year_match else None
    semester = semester_match.group(1) if semester_match else None

    return year, semester


def get_meta(x):
    return pd.Series(extract_metadata(x))


df[["exam_date", "semester_info"]] = df["text"].apply(get_meta)

In [12]:
df[["paper", "exam_date", "semester_info"]].head(40)

,paper,exam_date,semester_info
0,Discrete_Mathematics_2013_Final_E.pdf,DECEMBER,SEMESTER 1 - YEAR 1
1,Discrete_Mathematics_2013_Final_E.pdf,None,None
2,Discrete_Mathematics_2013_Final_E.pdf,None,None
3,Discrete_Mathematics_2013_Final_E.pdf,None,None
4,Discrete_Mathematics_2013_Final_E.pdf,None,None
5,Discrete_Mathematics_2013_Final_E.pdf,None,None
6,Discrete_Mathematics_2013_Final_E.pdf,None,None
7,Discrete_Mathematics_2013_Final_E.pdf,None,None
8,Discrete_Mathematics_2013_Final_E.pdf,None,None
11,Discrete_Mathematics_2013_Final_MS.pdf,None,None


# Distinguish between exam papers and marking schemes


In [13]:
df.columns

Index(['paper', 'page', 'method', 'length', 'page_count', 'text', 'exam_date',
       'semester_info'],
      dtype='object')

In [14]:
df["exam_date"] = df.groupby("paper")["exam_date"].ffill().bfill()
df["semester_info"] = df.groupby("paper")["semester_info"].ffill().bfill()

In [15]:
df.columns.tolist()

['paper',
 'page',
 'method',
 'length',
 'page_count',
 'text',
 'exam_date',
 'semester_info']

In [16]:
# Extract year
df["year"] = (
    df["paper"]
      .str.extract(r"Discrete_Mathematics_(\d{4})", expand=False)
      .astype("Int64")
)

In [17]:
# Extract sitting (Final / Repeat)
df["sitting"] = df["paper"].str.extract(r"_(Final|Repeat)_", expand=False)

In [18]:
# Convert E / MS into readable type
df["doc_type"] = df["paper"].str.extract(r"_(E|MS)\.pdf$", expand=False)
df["doc_type"] = df["doc_type"].map({
    "E": "Exam",
    "MS": "Marking Scheme"
})

In [19]:
papers = df[[
    "paper",
    "year",
    "sitting",
    "doc_type",
    "exam_date",
    "semester_info",
    "page_count"
]].drop_duplicates()

papers.sort_values(["year","sitting","doc_type"]).head(20)

,paper,year,sitting,doc_type,exam_date,semester_info,page_count
0,Discrete_Mathematics_2013_Final_E.pdf,2013,Final,Exam,DECEMBER,SEMESTER 1 - YEAR 1,12
11,Discrete_Mathematics_2013_Final_MS.pdf,2013,Final,Marking Scheme,DECEMBER,SEMESTER 1 - YEAR 1,21
32,Discrete_Mathematics_2014_Final_E.pdf,2014,Final,Exam,DECEMBER,SEMESTER 1 - YEAR 1,10
42,Discrete_Mathematics_2016_Final_E.pdf,2016,Final,Exam,DECEMBER,SEMESTER 1 - YEAR 1,12
54,Discrete_Mathematics_2017_Final_E.pdf,2017,Final,Exam,DECEMBER,SEMESTER 1 - YEAR 1,15
68,Discrete_Mathematics_2017_Repeat_E.pdf,2017,Repeat,Exam,DECEMBER,SEMESTER 1 - YEAR 1,11
79,Discrete_Mathematics_202122_Final_E.pdf,2021,Final,Exam,DECEMBER,SEMESTER 1 - YEAR 1,14
93,Discrete_Mathematics_202223_Final_E.pdf,2022,Final,Exam,DECEMBER,SEMESTER 1 - YEAR 1,7
100,Discrete_Mathematics_202223_Final_MS.pdf,2022,Final,Marking Scheme,DECEMBER,SEMESTER 1 - YEAR 1,7
107,Discrete_Mathematics_202223_Repeat_E.pdf,2022,Repeat,Exam,AUGUST,SEMESTER 1 - YEAR 1,7


# Fix OCR words

In [20]:
ms_2013 = df[df["paper"] == "Discrete_Mathematics_2013_Final_MS.pdf"]
print(ms_2013.iloc[2]["text"])

WATERFORD INSTITUTE OF TECHNOLOGY
OUTLINE MODEL ANSWERS AND MARKING SCHEME

Course: BScAppliedComputing & Computer Forensics Page. of Z|

& Entertainment Systems

Question — Subject Discrete Maths
Examiner Ms. Anne Daly Walsh Total |

ee ee
IRF Cr) CO3J4AVGi2) QA) |
| (goG@arpo
a ee ee
og
fe Ne
aa)

| S ucvalonce, Relahon must be
! | SO MO E XJ Wwwls : a trowarhwe.. pS

Refkaxire : es (OO) AL), 2), G3) S|
as eR, Vee A is tue |

ee Summetpic.+ Yes (0,3), (3,0) po
ee (ha) / 240) _
lds tag eA, if Gayf ten Goe R
| ig ae
ee

| VMeupe fil awereezyeR | |
7 they faayer os tue |

Oa VE ‘S hue
|
(OO) (0,3) -%6 (93)
> Equuvvalone twolahor. 5


In [21]:
# I've found the words that are spelt wrong in the marking scheme that did not come out perfect when using OCR so I looked at them and have figured out what the word is actually meant to be so I am fixing that by making a list and then a function to replace the misspelled words so then it'll be easier to match the answer to the question.

OCR_WORD_FIXES = {
    "questiun": "question",
    "subjevt": "subject",
    "entertaroment": "entertainment",
    "eutertainment": "entertainment",
    "relahon": "relation",
    "refkaxire": "reflexive",
    "summetpic": "symmetric",
    "symmetpic": "symmetric",
    "transihve": "transitive",
    "equuvvalone": "equivalence",
    "equivvalone": "equivalence",
    "ucvalonce": "equivalence",
    "twolahor": "relation",
    "distelutwe": "distributive",
    "distecbubae": "distributive",
    "cmplemeut": "complement",
    "complait": "complement",
    "saplemout": "complement",
    "jautology": "tautology",
    "implicaby": "implication",
}
def fix_ocr_words(text):
    if not text:
        return ""

    text = text.lower()
    text = re.sub(r"\s+", " ", text)

    for wrong, correct in OCR_WORD_FIXES.items():
        text = re.sub(rf"\b{re.escape(wrong)}\b", correct, text)

    return text.strip()

In [22]:
df["text_clean"] = df["text"].apply(fix_ocr_words)

In [23]:
ms_2013_clean = df[df["paper"] == "Discrete_Mathematics_2013_Final_MS.pdf"]
print(ms_2013_clean.iloc[2]["text_clean"])

waterford institute of technology outline model answers and marking scheme course: bscappliedcomputing & computer forensics page. of z| & entertainment systems question — subject discrete maths examiner ms. anne daly walsh total | ee ee irf cr) co3j4avgi2) qa) | | (gog@arpo a ee ee og fe ne aa) | s equivalence, relation must be ! | so mo e xj wwwls : a trowarhwe.. ps reflexive : es (oo) al), 2), g3) s| as er, vee a is tue | ee symmetric.+ yes (0,3), (3,0) po ee (ha) / 240) _ lds tag ea, if gayf ten goe r | ig ae ee | vmeupe fil awereezyer | | 7 they faayer os tue | oa ve ‘s hue | (oo) (0,3) -%6 (93) > equivalence relation. 5


# Identify question structure (A), (i), (ii), marks

In [24]:
df.head(5)

,paper,page,method,length,page_count,text,exam_date,semester_info,year,sitting,doc_type,text_clean
0,Discrete_Mathematics_2013_Final_E.pdf,1,text,740,12,1 \n \n WIT \n \n \n \nBACHELOR OF SC...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,1 wit bachelor of science (hons) in - applied ...
1,Discrete_Mathematics_2013_Final_E.pdf,2,text,924,12,2 \n \n \n \nQuestion 1 \n \n(a) \nThe univers...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,"2 question 1 (a) the universal set is {1, 2, 3..."
2,Discrete_Mathematics_2013_Final_E.pdf,3,text,632,12,3 \n \n \n \nQuestion 1 continued \n(c) \n \n(...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,3 question 1 continued (c) (i) investigate usi...
3,Discrete_Mathematics_2013_Final_E.pdf,5,text,860,12,5 \n \nQuestion 2 continued \n \n(b) \n(1) Us...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,5 question 2 continued (b) (1) using the laws ...
4,Discrete_Mathematics_2013_Final_E.pdf,6,text,1072,12,6 \n \nQuestion 3 \n \n(a) \n(i) \nHow many d...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,6 question 3 (a) (i) how many different ways a...


In [25]:
import re

df["question"] = df["text"].str.contains(r"Question\s+\d+", regex=True)
df["letter"] = df["text"].str.contains(r"\([a-z]\)", regex=True)
df["roman"] = df["text"].str.contains(r"\((i|ii|iii|iv|v)\)", regex=True)
df["marks"] = df["text"].str.contains(r"\(\s*\d+\s*marks?\s*\)", regex=True, case=False)
df["total"] = df["text"].str.contains(r"Total\s+\d+\s+Marks", regex=True, case=False)
df[[
    "paper",
    "page",
    "question",
    "letter",
    "roman",
    "marks",
    "total"
]].head(20)


/var/folders/sd/2991h4j52_q901zgt_52t4w40000gn/T/ipykernel_83500/587877511.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["roman"] = df["text"].str.contains(r"\((i|ii|iii|iv|v)\)", regex=True)


,paper,page,question,letter,roman,marks,total
0,Discrete_Mathematics_2013_Final_E.pdf,1,False,False,False,False,False
1,Discrete_Mathematics_2013_Final_E.pdf,2,True,True,True,True,False
2,Discrete_Mathematics_2013_Final_E.pdf,3,True,True,True,True,True
3,Discrete_Mathematics_2013_Final_E.pdf,5,True,True,True,True,True
4,Discrete_Mathematics_2013_Final_E.pdf,6,True,True,True,True,True
5,Discrete_Mathematics_2013_Final_E.pdf,7,True,True,True,True,False
6,Discrete_Mathematics_2013_Final_E.pdf,8,True,True,True,True,False
7,Discrete_Mathematics_2013_Final_E.pdf,9,True,True,True,True,True
8,Discrete_Mathematics_2013_Final_E.pdf,10,False,False,False,False,False
11,Discrete_Mathematics_2013_Final_MS.pdf,1,False,False,False,False,False


# Topic keywords

In [26]:
topics = {

    "Propositional Logic": [
        "truth table",
        "tautology",
        "contradiction",
        "logical equivalence",
        "well formed",
        "wff",
        "implication",
        "de morgan",
        "absorption",
        "boolean expression",
        "simplify",
        "∨", "∧", "¬", "→"
    ],

    "Predicate Logic / Quantifiers": [
        "∀",
        "∃",
        "for all",
        "there exists",
        "x ∈",
        "domain",
        "predicate"
    ],

    "Relations": [
        "relation",
        "reflexive",
        "symmetric",
        "transitive",
        "equivalence relation",
        "equivalence class",
        "antisymmetric",
        "asymmetric",
        "irreflexive",
        "digraph"
    ],

    "Counting / Combinatorics": [
        "subset",
        "cardinality",
        "|p(",
        "binomial",
        "bit string",
        "weight",
        "inclusion-exclusion",
        "lattice path",
        "permutation",
        "combination",
        "committee",
        "choose"
    ],

    "Sequences": [
        "arithmetic progression",
        "geometric progression",
        "recursive definition",
        "closed form",
        "induction",
        "partial sum",
        "common difference",
        "common ratio"
    ],

    "Functions": [
        "injective",
        "surjective",
        "onto",
        "one-to-one",
        "inverse function",
        "composition",
        "f(x)",
        "g(x)"
    ],

    "Graph Theory": [
        "degree sequence",
        "simple graph",
        "eulerian",
        "hamiltonian",
        "girth",
        "bipartite",
        "adjacency matrix",
        "vertex",
        "edge"
    ],

    "Python Programming": [
        "def ",
        "return ",
        "range(",
        "print(",
        ".union(",
        ".intersection(",
        ".difference(",
        ".issubset(",
        ".isdigit(",
        ".isalpha(",
        " for ",
        " in range",
        "set(",
        "list(",
        "//",
        "%"
    ]
}


In [27]:
def main_topics(text):
    text = text.lower()
    found = []

    for topic, words in topics.items():
        for w in words:
            if w.lower() in text:
                found.append(topic)
                break

    return found

df["topics"] = df["text"].apply(main_topics)

In [28]:
df[["paper", "page", "topics"]].head(20)

,paper,page,topics
0,Discrete_Mathematics_2013_Final_E.pdf,1,[]
1,Discrete_Mathematics_2013_Final_E.pdf,2,"[Propositional Logic, Relations]"
2,Discrete_Mathematics_2013_Final_E.pdf,3,"[Propositional Logic, Predicate Logic / Quanti..."
3,Discrete_Mathematics_2013_Final_E.pdf,5,"[Propositional Logic, Predicate Logic / Quanti..."
4,Discrete_Mathematics_2013_Final_E.pdf,6,"[Propositional Logic, Counting / Combinatorics..."
5,Discrete_Mathematics_2013_Final_E.pdf,7,[Graph Theory]
6,Discrete_Mathematics_2013_Final_E.pdf,8,[Graph Theory]
7,Discrete_Mathematics_2013_Final_E.pdf,9,"[Graph Theory, Python Programming]"
8,Discrete_Mathematics_2013_Final_E.pdf,10,"[Propositional Logic, Python Programming]"
11,Discrete_Mathematics_2013_Final_MS.pdf,1,[]


Removing "\n" from exam papers

In [29]:
df["text"] = df["text"].str.replace("\n", " ", regex=False)

In [30]:
print(df["text"].iloc[0][:1000])

1              WIT        BACHELOR OF SCIENCE (HONS) IN  - APPLIED COMPUTING   - COMPUTER FORENSICS & SECURITY  - ENTERTAINMENT SYSTEMS       EXAMINATION:  DISCRETE MATHEMATICS   (COMMON MODULE)  SEMESTER 1 - YEAR 1      DECEMBER 2013        DURATION:  2 HOURS          INTERNAL EXAMINER:  MS ANNE DALY WALSH    DATE:    16 DECEMBER, 2013.                TIME:   16.45 PM                VENUE:   MAIN HALL  EXTERNAL EXAMINERS:  DR FLAITHRÍ NEFF          DR ANTHONY KEANE          PROF M-TAHAR KECHADI      INSTRUCTIONS TO CANDIDATES    1. ANSWER THREE QUESTIONS.   2. TOTAL MARK IS 150.  3. MARKS MAY BE LOST IF ALL WORK IS NOT SHOWN CLEARLY         MATERIALS REQUIRED    1.  NEW MATHEMATICS TABLES.        WATERFORD INSTITUTE OF TECHNOLOGY


In [31]:
df.head(20)

,paper,page,method,length,page_count,text,exam_date,semester_info,year,sitting,doc_type,text_clean,question,letter,roman,marks,total,topics
0,Discrete_Mathematics_2013_Final_E.pdf,1,text,740,12,1 WIT BACHELOR OF SCIENCE ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,1 wit bachelor of science (hons) in - applied ...,False,False,False,False,False,[]
1,Discrete_Mathematics_2013_Final_E.pdf,2,text,924,12,2 Question 1 (a) The universal set ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,"2 question 1 (a) the universal set is {1, 2, 3...",True,True,True,True,False,"[Propositional Logic, Relations]"
2,Discrete_Mathematics_2013_Final_E.pdf,3,text,632,12,3 Question 1 continued (c) (i) Inv...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,3 question 1 continued (c) (i) investigate usi...,True,True,True,True,True,"[Propositional Logic, Predicate Logic / Quanti..."
3,Discrete_Mathematics_2013_Final_E.pdf,5,text,860,12,5 Question 2 continued (b) (1) Using t...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,5 question 2 continued (b) (1) using the laws ...,True,True,True,True,True,"[Propositional Logic, Predicate Logic / Quanti..."
4,Discrete_Mathematics_2013_Final_E.pdf,6,text,1072,12,6 Question 3 (a) (i) How many differe...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,6 question 3 (a) (i) how many different ways a...,True,True,True,True,True,"[Propositional Logic, Counting / Combinatorics..."
5,Discrete_Mathematics_2013_Final_E.pdf,7,text,345,12,7 Question 4 (a) (i) Draw the graph ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,7 question 4 (a) (i) draw the graph with five ...,True,True,True,True,False,[Graph Theory]
6,Discrete_Mathematics_2013_Final_E.pdf,8,text,556,12,8 Question 4 continued (b) (i) Invest...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,8 question 4 continued (b) (i) investigate the...,True,True,True,True,False,[Graph Theory]
7,Discrete_Mathematics_2013_Final_E.pdf,9,text,368,12,9 Question 4 continued (iii) Using Fi...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,9 question 4 continued (iii) using fig 3 above...,True,True,True,True,True,"[Graph Theory, Python Programming]"
8,Discrete_Mathematics_2013_Final_E.pdf,10,text,656,12,10 PLEASE ASK FOR THE NEW MATHEMATICS TABLE...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,10 please ask for the new mathematics tables l...,False,False,False,False,False,"[Propositional Logic, Python Programming]"
11,Discrete_Mathematics_2013_Final_MS.pdf,1,ocr,405,21,Sp ¢ SomesteR LL - 2015 WATERFORD INSTITUTE O...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Marking Scheme,sp ¢ somester ll - 2015 waterford institute of...,False,False,False,False,False,[]


In [32]:
df.shape

(168, 18)

In [33]:
df.columns

Index(['paper', 'page', 'method', 'length', 'page_count', 'text', 'exam_date',
       'semester_info', 'year', 'sitting', 'doc_type', 'text_clean',
       'question', 'letter', 'roman', 'marks', 'total', 'topics'],
      dtype='object')

In [34]:
df.head()

,paper,page,method,length,page_count,text,exam_date,semester_info,year,sitting,doc_type,text_clean,question,letter,roman,marks,total,topics
0,Discrete_Mathematics_2013_Final_E.pdf,1,text,740,12,1 WIT BACHELOR OF SCIENCE ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,1 wit bachelor of science (hons) in - applied ...,False,False,False,False,False,[]
1,Discrete_Mathematics_2013_Final_E.pdf,2,text,924,12,2 Question 1 (a) The universal set ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,"2 question 1 (a) the universal set is {1, 2, 3...",True,True,True,True,False,"[Propositional Logic, Relations]"
2,Discrete_Mathematics_2013_Final_E.pdf,3,text,632,12,3 Question 1 continued (c) (i) Inv...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,3 question 1 continued (c) (i) investigate usi...,True,True,True,True,True,"[Propositional Logic, Predicate Logic / Quanti..."
3,Discrete_Mathematics_2013_Final_E.pdf,5,text,860,12,5 Question 2 continued (b) (1) Using t...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,5 question 2 continued (b) (1) using the laws ...,True,True,True,True,True,"[Propositional Logic, Predicate Logic / Quanti..."
4,Discrete_Mathematics_2013_Final_E.pdf,6,text,1072,12,6 Question 3 (a) (i) How many differe...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,6 question 3 (a) (i) how many different ways a...,True,True,True,True,True,"[Propositional Logic, Counting / Combinatorics..."


In [35]:
df.to_csv("cleaned_discrete_maths_exams.csv", index=False)

In [36]:
import os
os.getcwd()

'/Users/jorjaholland/Desktop/DM2CA1/DM2CA1/notebooks'